# POYO-MP Linear Probe — Lick Classification
Zero-shot linear probe on top of a frozen POYO-MP backbone to classify
7 lick types from Purkinje cell spike trains.

**Datasets required (add via *Add data* in the sidebar):**
- `rat-data` — contains `1D_1.mat` and `1D_2.mat`
- `poyo-mp-checkpoint` — contains `poyo_mp.ckpt`

In [1]:
# Install torch_brain (takes ~1 min on first run)
!pip install git+https://github.com/neuro-galaxy/torch_brain.git -q

#V3


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires typeguard<5,>=4, but you have typeguard 2.13.3 which is incompatible.
inflect 7.5.0 requires typeguard>=4.0.1, but you have typeguard 2.13.3 which is incompatible.


In [2]:
import os, random, warnings
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

from torch_brain.models import POYO
from torch_brain.registry import ModalitySpec, DataType

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on: {device}')

# ── Reproducibility ──────────────────────────────────────────────────────────
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ── Paths (Kaggle dataset mount points) ──────────────────────────────────────
MAT_PATHS  = [
    '/kaggle/input/datasets/rahymfaisalkhan/rat-data/1D_1.mat',
    '/kaggle/input/datasets/rahymfaisalkhan/rat-data/1D_2.mat',
]
CKPT_PATH  = '/kaggle/input/datasets/mahama5if/poyo-weights/poyo_mp.ckpt'

# ── Hyperparameters ───────────────────────────────────────────────────────────
CONFIG = {
    'epochs':              30,
    'batch_size':          32,
    'lr':                  1e-3,
    'test_size':           0.2,
    'val_size':            0.1,
    'use_weighted_sampler': True,
}

LABEL_NAMES = [
    'groom',
    'inner_tube_success', 'inner_tube_fail',
    'outer_edge_success', 'outer_edge_fail',
    'under_tube_success', 'under_tube_fail',
]
N_CLASSES = 7

# Time window around each lick onset (seconds)
WINDOW_PRE  = 0.20
WINDOW_POST = 0.40
MAX_SPIKES  = 512


Running on: cuda


## Data Loading

In [3]:
def _find_spike_pair(refs, lick_key):
    """Find the two Purkinje cells with the most spikes in the lick window."""
    onsets = refs[f'{lick_key}/time_onset'][()].flatten()
    lick_start, lick_end = onsets[0], onsets[-1]
    all_keys = [
        k for k in refs
        if hasattr(refs[k], 'keys') and 'SS_time' in refs[k] and 'CS_time' in refs[k]
    ]
    scored = []
    for k in all_keys:
        ss = refs[f'{k}/SS_time'][()].flatten()
        scored.append((k, int(((ss >= lick_start) & (ss <= lick_end)).sum()), ss[0]))
    scored.sort(key=lambda t: -t[1])

    if scored[0][1] == 0:
        ref_start = scored[0][2]
        rescored = []
        for k, _, ss0 in scored:
            ss = refs[f'{k}/SS_time'][()].flatten()
            off = ref_start - ss0
            ov = int((((ss + off) >= lick_start) & ((ss + off) <= lick_end)).sum())
            rescored.append((k, off, ov))
        rescored.sort(key=lambda t: -t[2])
        k1, o1 = rescored[0][0], rescored[0][1]
        k2 = rescored[1][0] if len(rescored) > 1 else k1
        o2 = rescored[1][1] if len(rescored) > 1 else o1
        return k1, o1, k2, o2

    k1, o1 = scored[0][0], 0.0
    abs_start = refs[f'{k1}/SS_time'][()].flatten()[0]
    remaining = scored[1:]
    if remaining:
        rel = []
        for k, dov, ss0 in remaining:
            if dov > 0:
                rel.append((k, 0.0, dov))
            else:
                ss = refs[f'{k}/SS_time'][()].flatten()
                off = abs_start - ss0
                ov = int((((ss + off) >= lick_start) & ((ss + off) <= lick_end)).sum())
                rel.append((k, off, ov))
        rel.sort(key=lambda t: -t[2])
        k2, o2 = rel[0][0], rel[0][1]
    else:
        k2, o2 = k1, 0.0
    return k1, o1, k2, o2


def _load_session_spikes(f, lick_key):
    """
    Returns per-lick spike token lists.
    spike_tokens[i] = dict with:
        unit_indices : int array  (0=SS1, 1=SS2, 2=CS1, 3=CS2)
        timestamps   : float array (seconds, relative to lick onset)
    """
    refs = f['#refs#']
    lick_grp = refs[lick_key]
    y = lick_grp['tag_lick'][()].flatten().astype(int) - 1
    onsets = lick_grp['time_onset'][()].flatten()

    k1, o1, k2, o2 = _find_spike_pair(refs, lick_key)
    spike_trains = {
        0: refs[f'{k1}/SS_time'][()].flatten() + o1,
        1: refs[f'{k2}/SS_time'][()].flatten() + o2,
        2: refs[f'{k1}/CS_time'][()].flatten() + o1,
        3: refs[f'{k2}/CS_time'][()].flatten() + o2,
    }

    spike_tokens = []
    for onset in onsets:
        t0, t1 = onset - WINDOW_PRE, onset + WINDOW_POST
        units, times = [], []
        for uid, train in spike_trains.items():
            mask = (train >= t0) & (train <= t1)
            ts = (train[mask] - onset).astype(np.float32)
            units.extend([uid] * ts.size)
            times.extend(ts.tolist())
        order = np.argsort(times)
        spike_tokens.append({
            'unit_indices': np.array(units, dtype=np.int64)[order],
            'timestamps':   np.array(times, dtype=np.float32)[order],
        })

    return spike_tokens, y


def load_all_data(mat_paths):
    all_tokens, all_y = [], []
    for path in mat_paths:
        with h5py.File(path, 'r') as f:
            refs = f['#refs#']
            lick_keys = sorted(
                [k for k in refs if hasattr(refs[k], 'keys') and 'tag_lick' in refs[k]],
                key=lambda k: float(refs[f'{k}/time_onset'][()].flatten()[0]),
            )
            print(f'  {len(lick_keys)} session(s) in {os.path.basename(path)}')
            for lk in lick_keys:
                toks, yy = _load_session_spikes(f, lk)
                all_tokens.extend(toks)
                all_y.extend(yy)
    return all_tokens, np.array(all_y, dtype=np.int64)


print('Loading data...')
all_tokens, all_y = load_all_data(MAT_PATHS)
print(f'Total licks: {len(all_y)}')
print(f'Class distribution: {dict(zip(LABEL_NAMES, np.bincount(all_y, minlength=N_CLASSES)))}')


Loading data...
  3 session(s) in 1D_1.mat
  5 session(s) in 1D_2.mat
Total licks: 14002
Class distribution: {'groom': np.int64(2522), 'inner_tube_success': np.int64(652), 'inner_tube_fail': np.int64(353), 'outer_edge_success': np.int64(52), 'outer_edge_fail': np.int64(2396), 'under_tube_success': np.int64(1783), 'under_tube_fail': np.int64(6244)}


## Dataset & DataLoaders

In [4]:
class PoyoLickDataset(Dataset):
    def __init__(self, tokens, y, max_spikes=MAX_SPIKES):
        self.tokens    = tokens
        self.y         = torch.from_numpy(y)
        self.max_spikes = max_spikes

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        tok = self.tokens[i]
        n   = len(tok['timestamps'])
        L   = min(n, self.max_spikes)

        unit_idx = torch.zeros(self.max_spikes, dtype=torch.long)
        times    = torch.zeros(self.max_spikes, dtype=torch.float32)
        mask     = torch.zeros(self.max_spikes, dtype=torch.bool)

        unit_idx[:L] = torch.from_numpy(tok['unit_indices'][:L])
        times[:L]    = torch.from_numpy(tok['timestamps'][:L])
        mask[:L]     = True

        n_latents = 24
        lat_idx   = torch.arange(n_latents, dtype=torch.long)
        lat_times = torch.linspace(-WINDOW_PRE, WINDOW_POST, n_latents)

        # POYO's forward() requires output tokens — use a single dummy output at t=0
        # We intercept the latents before the decoder, so this is never actually used.
        out_times   = torch.zeros(1, dtype=torch.float32)
        out_sess    = torch.zeros(1, dtype=torch.long)

        return {
            'input_unit_index':   unit_idx,
            'input_timestamps':   times,
            'input_mask':         mask,
            'latent_index':       lat_idx,
            'latent_timestamps':  lat_times,
            'output_timestamps':  out_times,
            'output_session_idx': out_sess,
        }, self.y[i]


def collate_fn(batch):
    inputs = {k: torch.stack([b[0][k] for b in batch]) for k in batch[0][0]}
    labels = torch.stack([b[1] for b in batch])
    return inputs, labels


# ── Train / val / test split ─────────────────────────────────────────────────
idx = np.arange(len(all_y))
idx_tv, idx_test = train_test_split(
    idx, test_size=CONFIG['test_size'], stratify=all_y, random_state=42)
idx_train, idx_val = train_test_split(
    idx_tv,
    test_size=CONFIG['val_size'] / (1 - CONFIG['test_size']),
    stratify=all_y[idx_tv], random_state=42)

def subset(idxs):
    return [all_tokens[i] for i in idxs], all_y[idxs]

tr_tok, tr_y = subset(idx_train)
va_tok, va_y = subset(idx_val)
te_tok, te_y = subset(idx_test)

cnt            = np.bincount(tr_y, minlength=N_CLASSES).astype(float)
sample_weights = torch.tensor((1.0 / cnt[tr_y]).astype(np.float32))
sampler        = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

tr_ds = PoyoLickDataset(tr_tok, tr_y)
va_ds = PoyoLickDataset(va_tok, va_y)
te_ds = PoyoLickDataset(te_tok, te_y)

tr_ld = DataLoader(tr_ds, batch_size=CONFIG['batch_size'], sampler=sampler,  collate_fn=collate_fn)
va_ld = DataLoader(va_ds, batch_size=CONFIG['batch_size'], shuffle=False,    collate_fn=collate_fn)
te_ld = DataLoader(te_ds, batch_size=CONFIG['batch_size'], shuffle=False,    collate_fn=collate_fn)

print(f'Train: {len(tr_ds)}  Val: {len(va_ds)}  Test: {len(te_ds)}')


Train: 9800  Val: 1401  Test: 2801


## Model

In [5]:
class POYOEncoder(POYO):
    """
    Thin POYO subclass that short-circuits the decoder and returns
    mean-pooled latents instead of decoder outputs.
    The readout layer is constructed but never used.
    """
    def forward(
        self,
        *,
        input_unit_index,
        input_timestamps,
        input_token_type,
        input_mask=None,
        latent_index,
        latent_timestamps,
        **kwargs,   # absorbs output_* kwargs without error
    ):
        # ── encode ──────────────────────────────────────────────────────────
        inputs  = self.unit_emb(input_unit_index) + self.token_type_emb(input_token_type)
        in_temb = self.rotary_emb(input_timestamps)

        latents  = self.latent_emb(latent_index)
        lat_temb = self.rotary_emb(latent_timestamps)

        latents = latents + self.enc_atn(latents, inputs, lat_temb, in_temb, input_mask)
        latents = latents + self.enc_ffn(latents)

        # ── process ─────────────────────────────────────────────────────────
        for self_attn, self_ff in self.proc_layers:
            latents = latents + self.dropout(self_attn(latents, lat_temb))
            latents = latents + self.dropout(self_ff(latents))

        # ── pool and return (skip decoder entirely) ─────────────────────────
        return latents.mean(dim=1)   # (batch, dim)


class PoyoLickClassifier(nn.Module):
    def __init__(self, backbone_dim: int, n_classes: int = N_CLASSES):
        super().__init__()
        self.backbone = None   # set after weight loading
        self.head     = nn.Linear(backbone_dim, n_classes)

    def forward(self, inputs):
        emb = self.backbone(
            input_unit_index  = inputs['input_unit_index'],
            input_timestamps  = inputs['input_timestamps'],
            input_token_type  = torch.zeros_like(inputs['input_unit_index']),
            input_mask        = inputs['input_mask'],
            latent_index      = inputs['latent_index'],
            latent_timestamps = inputs['latent_timestamps'],
        )   # (batch, dim)  — decoder skipped by POYOEncoder
        return self.head(emb)


## Load POYO-MP Backbone

In [6]:
# Build a dummy ModalitySpec so POYO constructs its (unused) readout layer
_dummy_spec = ModalitySpec(
    id=0,
    dim=128,                    # matches backbone dim — readout is square, weights ignored
    type=DataType.CONTINUOUS,
    timestamp_key='timestamps',
    value_key='values',
    loss_fn=lambda x, y: x,  # dummy — never called
)

backbone = POYOEncoder(
    sequence_length      = WINDOW_PRE + WINDOW_POST,  # 0.6 s
    readout_spec         = _dummy_spec,                # required by POYO.__init__
    latent_step          = 0.25,
    num_latents_per_step = 32,
    dim                  = 128,
    depth                = 24,
    dim_head             = 64,
    cross_heads          = 4,
    self_heads           = 8,
    ffn_dropout          = 0.0,
    lin_dropout          = 0.0,
    atn_dropout          = 0.0,
    emb_init_scale       = 0.02,
    t_min                = 1e-4,
    t_max                = 4.0,
)

ckpt  = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
state = ckpt.get('state_dict', ckpt)
# Strip Lightning's 'model.' prefix; drop readout weights (our dummy spec != pretrained spec)
clean = {
    k.replace('model.', '', 1): v
    for k, v in state.items()
    if not k.replace('model.', '', 1).startswith('readout.')
}

missing, unexpected = backbone.load_state_dict(clean, strict=False)
print(f'Missing: {len(missing)}  Unexpected: {len(unexpected)}')

# Freeze all backbone weights — zero-shot linear probe only
for p in backbone.parameters():
    p.requires_grad = False
backbone.eval()

model = PoyoLickClassifier(backbone_dim=128, n_classes=N_CLASSES)
model.backbone = backbone
model = model.to(device)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')


Missing: 2  Unexpected: 0
Trainable params: 903


## Training

In [7]:
cw        = torch.tensor(1.0 / np.where(cnt > 0, cnt, 1.0), dtype=torch.float32)
cw       /= cw.sum()
criterion = nn.CrossEntropyLoss(weight=cw.to(device))
optimizer = torch.optim.AdamW(model.head.parameters(), lr=CONFIG['lr'])

best_val_acc    = 0.0
best_val_bac    = 0.0
best_head_state = None

for epoch in range(1, CONFIG['epochs'] + 1):
    # ── train ────────────────────────────────────────────────────────────────
    model.head.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    for inputs, labels in tr_ld:
        inputs = {k: v.to(device) for k, v in inputs.items()}
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(inputs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        tr_loss    += loss.item() * len(labels)
        tr_correct += (logits.argmax(1) == labels).sum().item()
        tr_total   += len(labels)

    # ── validate ─────────────────────────────────────────────────────────────
    model.eval()
    va_loss, va_correct, va_total = 0.0, 0, 0
    va_preds_all, va_gts_all = [], []    
    with torch.no_grad():
        for inputs, labels in va_ld:
            inputs = {k: v.to(device) for k, v in inputs.items()}
            labels = labels.to(device)
            logits = model(inputs)
            loss   = criterion(logits, labels)
            va_loss    += loss.item() * len(labels)
            va_correct += (logits.argmax(1) == labels).sum().item()
            va_total   += len(labels)
            va_preds_all.extend(logits.argmax(1).cpu().numpy()) 
            va_gts_all.extend(labels.cpu().numpy()) 

    tr_acc = tr_correct / tr_total
    va_acc = va_correct / va_total
    va_bac = balanced_accuracy_score(va_gts_all, va_preds_all)
    print(f'Epoch {epoch:3d}  |  '
          f'train loss {tr_loss/tr_total:.4f}  acc {tr_acc:.3f}  |  '
          f'val loss {va_loss/va_total:.4f}  acc {va_acc:.3f} bac {va_bac:.3f}')

    if va_bac > best_val_bac:
        best_val_bac    = va_bac
        best_val_acc    = va_acc  # track corresponding acc
        best_head_state = {k: v.clone() for k, v in model.head.state_dict().items()}

print(f'\nBest val acc: {best_val_acc:.4f}  |  Best val BAC: {best_val_bac:.4f}')


Epoch   1  |  train loss 67.1887  acc 0.162  |  val loss 52.0903  acc 0.021 bac 0.158
Epoch   2  |  train loss 9.5920  acc 0.177  |  val loss 58.1561  acc 0.019 bac 0.180
Epoch   3  |  train loss 10.6562  acc 0.168  |  val loss 54.0847  acc 0.020 bac 0.133
Epoch   4  |  train loss 7.4971  acc 0.179  |  val loss 35.2663  acc 0.127 bac 0.164
Epoch   5  |  train loss 8.9649  acc 0.170  |  val loss 70.2898  acc 0.039 bac 0.183
Epoch   6  |  train loss 8.7802  acc 0.182  |  val loss 33.6989  acc 0.046 bac 0.102
Epoch   7  |  train loss 8.7021  acc 0.184  |  val loss 44.7438  acc 0.009 bac 0.124
Epoch   8  |  train loss 8.2339  acc 0.191  |  val loss 138.9942  acc 0.004 bac 0.147
Epoch   9  |  train loss 7.7304  acc 0.196  |  val loss 49.7452  acc 0.007 bac 0.132
Epoch  10  |  train loss 7.8899  acc 0.188  |  val loss 108.2212  acc 0.011 bac 0.168
Epoch  11  |  train loss 7.9845  acc 0.197  |  val loss 72.2176  acc 0.020 bac 0.184
Epoch  12  |  train loss 8.6184  acc 0.190  |  val loss 37.99

## Evaluation

In [8]:
model.head.load_state_dict(best_head_state)
model.eval()

preds, gts = [], []
with torch.no_grad():
    for inputs, labels in te_ld:
        inputs = {k: v.to(device) for k, v in inputs.items()}
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        gts.extend(labels.numpy())

preds, gts = np.array(preds), np.array(gts)
te_acc = (preds == gts).mean()
te_bac = balanced_accuracy_score(gts, preds)
print(f'Test accuracy: {te_acc:.4f}')
print(f'Test BAC:      {te_bac:.4f}\n')

print(f"{'Class':<28} {'Acc':>6}  {'N':>5}")
print('-' * 44)
for c in range(N_CLASSES):
    m = gts == c
    if m.sum() > 0:
        print(f'{LABEL_NAMES[c]:<28} {(preds[m]==c).mean():.3f}   {m.sum():>5}')

Test accuracy: 0.0882
Test BAC:      0.1922

Class                           Acc      N
--------------------------------------------
groom                        0.246     505
inner_tube_success           0.769     130
inner_tube_fail              0.211      71
outer_edge_success           0.100      10
outer_edge_fail              0.000     479
under_tube_success           0.020     357
under_tube_fail              0.000    1249


## Few Shot Experiment 
Same frozen POYO backbone, but training head on only K examples per class.

In [27]:
from tabulate import tabulate

K_VALUES = [5, 10, 25, 50, 100]
FS_EPOCHS = 10
results = []

def few_shot_indices(labels, k):
    indices = []
    for c in range(N_CLASSES):
        class_idx = np.where(labels == c)[0]
        chosen = np.random.choice(class_idx, min(k, len(class_idx)), replace=False)
        indices.extend(chosen.tolist())
    return indices

for K in K_VALUES:
    set_seed(42)
    print(f"\nStarting K={K}...")

    # ── Split ─────────────────────────────────────────────────────────────
    fs_train_idx = few_shot_indices(all_y, K)
    remaining_idx = list(set(range(len(all_y))) - set(fs_train_idx))

    fs_tr_ds = PoyoLickDataset([all_tokens[i] for i in fs_train_idx], all_y[fs_train_idx])
    fs_te_ds = PoyoLickDataset([all_tokens[i] for i in remaining_idx], all_y[np.array(remaining_idx)])

    fs_tr_ld = DataLoader(fs_tr_ds, batch_size=min(32, len(fs_train_idx)), shuffle=True, collate_fn=collate_fn)
    fs_te_ld = DataLoader(fs_te_ds, batch_size=512, shuffle=False, collate_fn=collate_fn)

    # ── Fresh head ────────────────────────────────────────────────────────
    model.head = nn.Linear(model.head.in_features, N_CLASSES).to(device)
    fs_optimizer = torch.optim.AdamW(model.head.parameters(), lr=CONFIG['lr'])
    fs_criterion = nn.CrossEntropyLoss()

    # ── Train ─────────────────────────────────────────────────────────────
    fs_best_bac   = 0.0
    fs_best_state = None

    for epoch in range(1, FS_EPOCHS + 1):
        model.head.train()
        for inputs, labels in fs_tr_ld:
            inputs = {k: v.to(device) for k, v in inputs.items()}
            labels = labels.to(device)
            fs_optimizer.zero_grad()
            loss = fs_criterion(model(inputs), labels)
            loss.backward()
            fs_optimizer.step()

        model.eval()
        preds, gts = [], []
        with torch.no_grad():
            for inputs, labels in fs_tr_ld:
                inputs = {k: v.to(device) for k, v in inputs.items()}
                preds.extend(model(inputs).argmax(1).cpu().numpy())
                gts.extend(labels.numpy())
        bac = balanced_accuracy_score(gts, preds)
        if bac > fs_best_bac:
            fs_best_bac   = bac
            fs_best_state = {k: v.clone() for k, v in model.head.state_dict().items()}
        print(f"  Epoch {epoch:2d}/{FS_EPOCHS}  BAC {bac:.3f}", end='\r')

    print(f"  Training done. Best BAC: {fs_best_bac:.3f}        ")

    # ── Evaluate ──────────────────────────────────────────────────────────
    print(f"  Evaluating on {len(fs_te_ds)} test samples...")
    model.head.load_state_dict(fs_best_state)
    model.eval()

    preds, gts = [], []
    with torch.no_grad():
        for inputs, labels in fs_te_ld:
            inputs = {k: v.to(device) for k, v in inputs.items()}
            preds.extend(model(inputs).argmax(1).cpu().numpy())
            gts.extend(labels.numpy())

    preds, gts = np.array(preds), np.array(gts)
    te_acc = (preds == gts).mean()
    te_bac = balanced_accuracy_score(gts, preds)

    per_class = []
    per_class_n = []
    for c in range(N_CLASSES):
        m = gts == c
        acc = (preds[m] == c).mean() if m.sum() > 0 else 0.0
        per_class.append(round(acc, 3))
        per_class_n.append(int(m.sum()))

    results.append([K, K * N_CLASSES, round(te_acc, 4), round(te_bac, 4)] + per_class + per_class_n)

# ── Zero-shot row ─────────────────────────────────────────────────────────
print("\nAdding zero-shot row...")
model.head.load_state_dict(best_head_state)
model.eval()
zs_preds, zs_gts = [], []
with torch.no_grad():
    for inputs, labels in te_ld:
        inputs = {k: v.to(device) for k, v in inputs.items()}
        zs_preds.extend(model(inputs).argmax(1).cpu().numpy())
        zs_gts.extend(labels.numpy())

zs_preds, zs_gts = np.array(zs_preds), np.array(zs_gts)
zs_acc = (zs_preds == zs_gts).mean()
zs_bac = balanced_accuracy_score(zs_gts, zs_preds)
zs_per_class = []
zs_per_class_n = []
for c in range(N_CLASSES):
    m = zs_gts == c
    acc = (zs_preds[m] == c).mean() if m.sum() > 0 else 0.0
    zs_per_class.append(round(acc, 3))
    zs_per_class_n.append(int(m.sum()))
results.append(['Full', len(tr_ds), round(zs_acc, 4), round(zs_bac, 4)] + zs_per_class + zs_per_class_n)

# ── Print results ─────────────────────────────────────────────────────────
for row in results:
    K_label = f"K={row[0]}" if row[0] != 'Full' else "Full (Zero-Shot)"
    print(f"\n{'='*50}")
    print(f"{K_label}  |  Train N={row[1]}")
    print(f"Test accuracy: {row[2]:.4f}")
    print(f"Test BAC:      {row[3]:.4f}\n")
    print(f"{'Class':<28} {'Acc':>6}  {'N':>5}")
    print('-' * 44)
    for c in range(N_CLASSES):
        acc = row[4 + c]
        n   = row[4 + N_CLASSES + c]
        print(f"{LABEL_NAMES[c]:<28} {acc:.3f}   {n:>5}")


Starting K=5...
  Training done. Best BAC: 0.200        
  Evaluating on 13967 test samples...

Starting K=10...
  Training done. Best BAC: 0.186        
  Evaluating on 13932 test samples...

Starting K=25...
  Training done. Best BAC: 0.189        
  Evaluating on 13827 test samples...

Starting K=50...
  Training done. Best BAC: 0.194        
  Evaluating on 13652 test samples...

Starting K=100...
  Training done. Best BAC: 0.210        
  Evaluating on 13350 test samples...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



Adding zero-shot row...

K=5  |  Train N=35
Test accuracy: 0.3578
Test BAC:      0.1359

Class                           Acc      N
--------------------------------------------
groom                        0.000    2517
inner_tube_success           0.077     647
inner_tube_fail              0.000     348
outer_edge_success           0.000      47
outer_edge_fail              0.132    2391
under_tube_success           0.000    1778
under_tube_fail              0.742    6239

K=10  |  Train N=70
Test accuracy: 0.3475
Test BAC:      0.1245

Class                           Acc      N
--------------------------------------------
groom                        0.065    2512
inner_tube_success           0.000     642
inner_tube_fail              0.000     343
outer_edge_success           0.024      42
outer_edge_fail              0.052    2386
under_tube_success           0.000    1773
under_tube_fail              0.730    6234

K=25  |  Train N=175
Test accuracy: 0.1911
Test BAC:      0.1698
